# 10 — Block-Scaled Matrix Multiplication

**Ladder:** Scale-then-matmul (materialize) → Triton scale + tutorial-03 GEMM → (official) in-kernel FP8 / block scales fused into the $K$-loop.

**Study tip.** Master the scalar identity $(s_A s_B)AB$ first. Materializing full scaled $A,B$ teaches the cost; the real FP8 path fuses scales inside the $K$-loop. Then derive the quantization error bound — it explains why block scales exist.

Official: [Block-Scaled Matmul](https://triton-lang.org/main/getting-started/tutorials/10-block-scaled-matmul.html)

Primer: [`../03_matrix_multiplication/README.md`](../03_matrix_multiplication/README.md) · figures: [`docs/images/`](docs/images/)

![Whiteboard: scale-then-GEMM vs in-kernel block scales](docs/images/benchmark-block-scaled-matmul.png)

*Theory board — materialize scaled $A,B$ vs dequant with per-block scales inside the $K$-loop.*

---

# Part I — Mathematics

## 0. Scalar scaling and block scaling

### 0.1 The scalar identity

For $s_A,s_B\in\mathbb{R}$, $A\in\mathbb{R}^{M\times K}$,
$B\in\mathbb{R}^{K\times N}$:

$$
C=(s_AA)(s_BB)=(s_As_B)(AB).
\tag{0.1}
$$

*Proof.* Entrywise,

$$
\big[(s_AA)(s_BB)\big]_{mn}
=
\sum_k(s_Aa_{mk})(s_Bb_{kn})
=
s_As_B\sum_ka_{mk}b_{kn}
=
\big[(s_As_B)AB\big]_{mn}.
\qquad\square
$$

**Homogeneity.** Scaling inputs by constants factors out of the bilinear map. In
exact arithmetic (0.1) is an identity; in floating point the two sides differ
only by rounding order. This is what licenses "dequantize on the fly": the
scales can be applied before, during, or after the dot product.

### 0.2 Block scales

Suppose $A$ is stored quantized as $A_q$ with a scale for each
$(B_M\times B_K)$ tile:

$$
A_{p_m,t}=s^{A}_{p_m,t}\,A_q[p_m,t],
\qquad
p_m\in[G_M],\ t\in[G_K],
\tag{0.2}
$$

and similarly

$$
B_{t,p_n}=s^{B}_{t,p_n}\,B_q[t,p_n],
\qquad
t\in[G_K],\ p_n\in[G_N].
\tag{0.3}
$$

Here $s^{A}_{p_m,t},s^{B}_{t,p_n}$ are **scalars** (one per block), while
$A_q[p_m,t]$ is a matrix block. The contribution of $K$-panel $t$ to output tile
$(p_m,p_n)$ is

$$
\begin{aligned}
C_{p_m,p_n}
&=
\sum_{t=0}^{G_K-1}
\left(s^{A}_{p_m,t}A_q[p_m,t]\right)
\left(s^{B}_{t,p_n}B_q[t,p_n]\right)\\
&=
\sum_{t=0}^{G_K-1}
s^{A}_{p_m,t}\,s^{B}_{t,p_n}
\left(A_q[p_m,t]\,B_q[t,p_n]\right).
\end{aligned}
\tag{0.4}
$$

Because the scales are scalars for fixed $(p_m,t,p_n)$, they commute with the
matrix product and can be applied **after** each `tl.dot`:

$$
\mathrm{Acc}
\leftarrow
\mathrm{Acc}
+
\big(s^{A}_{p_m,t}s^{B}_{t,p_n}\big)
\operatorname{dot}\!\left(A_q[p_m,t],\,B_q[t,p_n]\right).
\tag{0.5}
$$

This is the entire algebra of a block-scaled GEMM. The kernel never materializes
$s_AA$ or $s_BB$; it loads one scale per panel and multiplies the FP32
accumulator.

### 0.3 FP8 formats

An FP8 value has 1 sign bit and 7 bits split between exponent and mantissa:

| Format | Exponent bits | Mantissa bits | Max finite | Unit roundoff $u=2^{-(p+1)}$ |
|---|---|---|---|---|
| E4M3 | 4 | 3 | $448$ | $2^{-4}=6.25\%$ |
| E5M2 | 5 | 2 | $57{,}344$ | $2^{-3}=12.5\%$ |

E4M3 has more precision but a smaller range; E5M2 has more range but less
precision. The choice is a range/precision trade-off, exactly like FP16 vs
BF16 but much coarser.

**Memory.** FP8 stores one byte per element, half of FP16. A block of $32$
elements needs one scale; if the scale is FP32 (4 bytes), the overhead is
$4/32=0.125$ bytes per element, so the effective size is $1.125$ bytes/element
— still $1.78\times$ smaller than FP16 and $3.56\times$ smaller than FP32.

### 0.4 Quantization error

Let $q_{\max}$ be the largest representable magnitude (e.g. $448$ for E4M3).
For a block with maximum magnitude $x_{\max}$, choose

$$
s=\frac{x_{\max}}{q_{\max}},
\qquad
q=\operatorname{round}\!\left(\frac{x}{s}\right),
\qquad
\hat x=s\,q .
\tag{0.6}
$$

**Proposition 1 (absolute error).** If no clipping occurs, i.e.
$|x|\le x_{\max}$, then

$$
|x-\hat x|\le\frac{s}{2}=\frac{x_{\max}}{2q_{\max}}.
\tag{0.7}
$$

*Proof.* Rounding to the nearest representable multiple of $s$ has error at most
$s/2$. $\square$

**Corollary 2 (relative error).** For $|x|\ge x_{\max}/q_{\max}$ (normal range),

$$
\frac{|x-\hat x|}{|x|}
\le
\frac{s/2}{|x|}
\le
\frac{1}{2}\cdot\frac{x_{\max}}{q_{\max}}\cdot\frac{q_{\max}}{x_{\max}}
=
\frac12,
\tag{0.8}
$$

which is loose; the floating-point relative error is at most
$u=2^{-(p+1)}$ for values in the same binade. The practical rule is: FP8 gives
about 2–3 decimal digits, so accumulators must be higher precision.

**Corollary 3 (error in the product).** Write $A_q=A+\Delta A$,
$B_q=B+\Delta B$ with $|\Delta A|\le s_A/2$, $|\Delta B|\le s_B/2$. Then

$$
A_qB_q
=
AB+\Delta A\,B+A\,\Delta B+\Delta A\,\Delta B,
\tag{0.9}
$$

so

$$
\|A_qB_q-AB\|_F
\le
\|\Delta A\|_F\|B\|_2+\|A\|_2\|\Delta B\|_F+\|\Delta A\|_F\|\Delta B\|_F .
\tag{0.10}
$$

The relative error is $O(s_A/\|A\|+s_B/\|B\|)$, i.e. inversely proportional to
the block's dynamic range. This is why smaller blocks (finer scales) improve
accuracy: each block has a smaller $x_{\max}$ relative to its typical values.

### 0.5 Why per-block scales beat per-tensor scales

With one scale for the whole matrix, a single outlier of magnitude $10^{3}$ sets
$s\approx10^{3}/448\approx2.2$. Values of magnitude $1$ then quantize to
$q=\operatorname{round}(1/2.2)=0$ — they are lost. With per-block scales, only
the outlier's block pays; the other blocks keep a small $s$ and retain their
small values.

Quantitatively, the relative error scales with

$$
\frac{x_{\max}^{\mathrm{block}}}{\|A^{\mathrm{block}}\|},
\tag{0.11}
$$

so partitioning by locality reduces the numerator without changing the
denominator. This is the MX/block-scaled format idea used in FP8 LLM inference.

### 0.6 Arithmetic intensity and roofline

FP8 doubles the bytes-per-element advantage:

$$
Q_{\mathrm{fp8}}\approx\frac12Q_{\mathrm{fp16}},
\qquad
\mathrm{AI}_{\mathrm{fp8}}\approx2\,\mathrm{AI}_{\mathrm{fp16}} .
\tag{0.12}
$$

Tensor cores also have higher FP8 throughput, so FP8 GEMMs can be both
bandwidth- and compute-faster. The limiting factor becomes the **scaling
overhead** (0.5): one multiply per $K$-panel per output tile, which is
negligible compared with $B_MB_NB_K$ MACs.

### 0.7 Running example

$M=256$, $K=128$, $N=512$, $s_A=0.5$, $s_B=2.0$. Then $s_As_B=1$, so

$$
C=(0.5A)(2B)=AB,
\tag{0.13}
$$

which the tests use to check that scaling is algebraically transparent.

### Theory board

![Whiteboard: scale-then-GEMM vs in-kernel block scales](docs/images/whiteboard-theory.jpg)

*Theory board — materialize scaled $A,B$ vs dequant with per-block scales inside the $K$-loop.*

# Part II — Proof

**Theorem (scalar scaling).** For scalars $s_A,s_B$,
$(s_AA)(s_BB)=(s_As_B)(AB)$.

*Proof.* Entrywise,
$\sum_k(s_Aa_{mk})(s_Bb_{kn})=s_As_B\sum_ka_{mk}b_{kn}$. $\square$

**Theorem (block scaling).** If $A_{p_m,t}=s^A_{p_m,t}A_q[p_m,t]$ and
$B_{t,p_n}=s^B_{t,p_n}B_q[t,p_n]$, then

$$
C_{p_m,p_n}
=
\sum_t s^A_{p_m,t}s^B_{t,p_n}
\left(A_q[p_m,t]B_q[t,p_n]\right).
$$

*Proof.* Substitute the two factorizations into the block product and pull the
scalars (constant for fixed $t$) out of each matrix product. $\square$

**Proposition (quantization error).** If no clipping occurs and rounding is to
the nearest multiple of $s$, then $|x-\hat x|\le s/2$.

*Proof.* The nearest representable value is within half a spacing of $x$.
$\square$

**Corollary (product error).** With $A_q=A+\Delta A$,
$B_q=B+\Delta B$,
$A_qB_q=AB+\Delta AB+A\Delta B+\Delta A\Delta B$.

*Proof.* Expand the product algebraically. $\square$

# Part III — Step-by-step proof

1. **Choose scales.** For each block, set $s=x_{\max}/q_{\max}$.
2. **Quantize.** $q=\operatorname{round}(x/s)$, store $q$ in FP8.
3. **Store metadata.** Keep one scale per block.
4. **Tile the output.** Use $B_M,B_N$ as in notebook 03.
5. **Loop over K-panels.** Load $A_q[t],B_q[t]$ and their scales
   $s^A_{p_m,t},s^B_{t,p_n}$.
6. **Dot.** Compute $A_q[t]B_q[t]$ in the tensor core.
7. **Dequantize the contribution.**
   $\mathrm{Acc}\leftarrow\mathrm{Acc}+s^As^B\,A_q[t]B_q[t]$.
8. **Repeat and store.** After the last panel, cast and store $C$.
9. **Error check.** Bound the result by
   $\|\Delta A\|\|B\|+\|A\|\|\Delta B\|+\|\Delta A\|\|\Delta B\|$.

# Part IV — Coding

The coding part follows the three-rung ladder used in the PDFs: **Method 01 — naive**, **Method 02 — kernel-based**, and **Method 03 — fused kernel-based**. Each method is runnable and is checked against the previous one.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import importlib.util

import pytest
import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")


try:
    _HERE = Path(__file__).resolve().parent
except NameError:  # Jupyter notebook: __file__ is undefined
    _HERE = Path.cwd()
_REPO = _HERE
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "03_matrix_multiplication" / "tutorial.py").exists():
        _REPO = _candidate
        break
else:
    raise FileNotFoundError(
        "cannot locate 03_matrix_multiplication from " + str(_HERE)
    )
_matmul_path = _REPO / "03_matrix_multiplication" / "tutorial.py"
_spec = importlib.util.spec_from_file_location("matmul_mod", _matmul_path)
_matmul_mod = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_matmul_mod)
matmul = _matmul_mod.matmul


### Method 01 — Naive (materialized scaled operands)

| Stage | Realization | Memory story |
|---|---|---|
| **Naive** | Materialize $\tilde A=s_AA$, $\tilde B=s_BB$, then $\tilde A\tilde B$ | Two full scaled tensors + GEMM |
| **Intermediate** | Same scaling in PyTorch; GEMM via tutorial-03 `matmul` | Same math; Triton GEMM body |
| **Triton sketch** | Elementwise `y=x*s` kernel, then grouped matmul | Separable teaching steps |
| **Official** | In-kernel per-block / FP8 scales next to each $K$-panel | No full dequant copies |

**Equivalence (scalars).** All notebook paths compute $(s_As_B)AB$ in exact
arithmetic; FP differences come only from rounding order.

### 3.1 Cost of materializing the scaled operands

The naive path allocates and writes

$$
\tilde A=s_AA\in\mathbb{R}^{M\times K},
\qquad
\tilde B=s_BB\in\mathbb{R}^{K\times N},
\tag{3.1}
$$

so its traffic is

$$
Q_{\mathrm{naive}}
=
\underbrace{MK}_{\text{read }A}
+\underbrace{MK}_{\text{write }\tilde A}
+\underbrace{KN}_{\text{read }B}
+\underbrace{KN}_{\text{write }\tilde B}
+\underbrace{MK\left\lceil N/B_N\right\rceil + KN\left\lceil M/B_M\right\rceil}_{\text{tiled GEMM reads}}
+\underbrace{MN}_{\text{write }C}
\ \text{elements}.
\tag{3.2}
$$

The fused path never writes $\tilde A,\tilde B$: it applies the scale inside the
tile loop, removing $MK+KN$ writes and the corresponding allocations. The
saving is modest for large GEMMs but decisive for small ones, where the
elementwise kernels' launch and traffic dominate.

### 3.2 Per-tensor versus per-block error

Let $\|A\|_{\max}$ be the global maximum and $a_{\mathrm{typ}}$ a typical
magnitude. A per-tensor scale gives a relative error of order

$$
\varepsilon_{\mathrm{tensor}}
\approx
u\,\frac{\|A\|_{\max}}{a_{\mathrm{typ}}},
\tag{3.3}
$$

whereas a per-block scale gives

$$
\varepsilon_{\mathrm{block}}
\approx
u\,\frac{\|A^{\mathrm{block}}\|_{\max}}{a^{\mathrm{block}}_{\mathrm{typ}}}.
\tag{3.4}
$$

If one block contains the outlier and the rest do not, the second ratio is
$O(1)$ while the first can be $10^{3}$ or more. That is the quantitative case for
block scaling.

### 3.3 Error metric

The natural measure is the relative Frobenius error

$$
\mathrm{rel.err}
=
\frac{\|C-\hat C\|_F}{\|C\|_F},
\qquad
\hat C=\sum_t s^{A}_{t}s^{B}_{t}\operatorname{dot}(A_{q,t},B_{q,t}).
\tag{3.5}
$$

A good block-scaled kernel keeps this below a few percent for E4M3 with block
size $32$. The test in this notebook checks the scalar case with
`atol=rtol=1e-2`, which is the right order of magnitude for FP16 inputs.

### 3.4 In-kernel dequantization

The official FP8 kernel performs, for each $K$-panel $t$,

$$
\begin{aligned}
A_t&=\operatorname{load}(A_q[\,:,\,tB_K{:}(t+1)B_K\,]),\\
B_t&=\operatorname{load}(B_q[\,tB_K{:}(t+1)B_K,\,:\,]),\\
s_A&=\operatorname{load}(s^{A}_{p_m,t}),\qquad
s_B=\operatorname{load}(s^{B}_{t,p_n}),\\
\mathrm{Acc}&\leftarrow\mathrm{Acc}+(s_As_B)\operatorname{dot}(A_t,B_t).
\end{aligned}
\tag{3.6}
$$

Only $O(1)$ scale values are loaded per panel; the data stay in FP8. The
accumulator is FP32 (or FP16 with care), because the products of quantized
values must be summed at higher precision.

**Block-size choice.** A block size of $32$ along $K$ is the MX standard: large
enough that the scale overhead is $1/32$ per element, small enough that the
dynamic range within a block is limited. Smaller blocks improve accuracy but
increase metadata traffic and the number of scale multiplies.

### 3.5 Memory arithmetic

For $M=K=N=4096$:

| Format | Data bytes | Scales (block 32) | Total |
|---|---|---|---|
| FP32 | $3\cdot4096^{2}\cdot4=192$ MiB | — | $192$ MiB |
| FP16 | $96$ MiB | — | $96$ MiB |
| FP8 E4M3 | $48$ MiB | $2\cdot(4096^{2}/32)\cdot4=4$ MiB | $52$ MiB |

FP8 with FP32 scales is about $1.85\times$ smaller than FP16. If the scales are
stored as E8M0 (one byte), the overhead drops to $1$ MiB, giving $49$ MiB.

### 3.6 Summary

| Quantity | Naive / materialized | Fused block-scaled |
|---|---|---|
| Extra tensors | $\tilde A,\tilde B$ | none |
| Scale reads | none | $O(G_MG_K+G_KG_N)$ |
| Data precision | FP16/FP32 | FP8 |
| Accumulator | FP32 | FP32 |
| Algebra | $(s_As_B)AB$ | $\sum_t s^A_ts^B_tA_tB_t$ |

The fused form is algebraically identical to the naive one by (0.4)–(0.5); it
differs only in **where** the scale is applied and **how many bytes** are moved.


In [ ]:
def naive_block_scaled_matmul_pseudo(a, b, scale_a, scale_b):
    """
    Materialize scaled A/B in memory, then matmul.

    Memory: two full scaled tensors + the GEMM.
    Fine for scalars; wasteful when scales are per-block and you could
    dequantize inside the GEMM tile loop (official FP8 path).
    """
    return torch.matmul(a * scale_a, b * scale_b)


### Method 02 — Kernel-based (separate scaling kernel)

The first code cell is the reference that scales in PyTorch and calls the tutorial-03 GEMM. The second keeps the scaling and the GEMM as two kernels: `scale_kernel` writes
$a s_A$ and $b s_B$ to temporary buffers, then the tutorial-03 matmul consumes
them. It is fully kernel-based, but it materializes the dequantized operands and
pays their HBM traffic.


In [ ]:
def block_scaled_matmul_ref(a, b, scale_a, scale_b):
    """
    Intermediate: still scale in PyTorch, but GEMM via Triton tutorial 03.
    """
    return matmul(a * scale_a, b * scale_b)


In [ ]:
@triton.jit
def scale_kernel(x_ptr, y_ptr, scale, n_elements, BLOCK_SIZE: tl.constexpr):
    """Elementwise y = x * scale — stand-in for on-the-fly dequant loads."""
    pid = tl.program_id(0)
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask)
    tl.store(y_ptr + offs, x * scale, mask=mask)


def triton_scale(x, scale):
    y = torch.empty_like(x)
    n = x.numel()
    grid = lambda meta: (triton.cdiv(n, meta["BLOCK_SIZE"]),)
    scale_kernel[grid](x, y, float(scale), n, BLOCK_SIZE=1024)
    return y


def block_scaled_matmul_triton(a, b, scale_a, scale_b):
    """
    Triton path: scale with a tiny kernel, then grouped matmul from tutorial 03.

    Real block-scaled FP8 kernels fuse the scale lookup into the K-loop loads;
    this keeps the teaching steps separable.
    """
    return matmul(triton_scale(a, scale_a), triton_scale(b, scale_b))


### Method 03 — Fused kernel-based (scale inside the GEMM)

The scale is applied to the FP32 accumulator after the $K$-panel dot product:

$$
\mathrm{Acc}\leftarrow\mathrm{Acc}
+s_As_B\operatorname{dot}(A_t,B_t).
$$

No scaled copies of $A$ or $B$ are written. The kernel below implements the
scalar case; replacing $s_As_B$ by a per-block load gives the production FP8
path.


In [ ]:
@triton.jit
def block_scaled_matmul_kernel(
    A_ptr,
    B_ptr,
    C_ptr,
    M,
    N,
    K,
    scale_a,
    scale_b,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """Method 03: scalar block scale applied to the FP32 accumulator."""
    pid = tl.program_id(0)
    num_n = tl.cdiv(N, BLOCK_N)
    pid_m = pid // num_n
    pid_n = pid % num_n

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    a_ptrs = A_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs = B_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k in range(0, tl.cdiv(K, BLOCK_K)):
        remaining = K - k * BLOCK_K
        a = tl.load(
            a_ptrs,
            mask=(offs_m[:, None] < M) & (offs_k[None, :] < remaining),
            other=0.0,
        )
        b = tl.load(
            b_ptrs,
            mask=(offs_k[:, None] < remaining) & (offs_n[None, :] < N),
            other=0.0,
        )
        acc += tl.dot(a, b)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    # Dequantize the whole tile at once: s_A and s_B are scalars here.
    acc = acc * (scale_a * scale_b)

    c = acc.to(C_ptr.dtype.element_ty)
    c_ptrs = C_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    tl.store(
        c_ptrs,
        c,
        mask=(offs_m[:, None] < M) & (offs_n[None, :] < N),
    )


def block_scaled_matmul_fused(
    a, b, scale_a, scale_b, BLOCK_M=64, BLOCK_N=64, BLOCK_K=32
):
    """Method 03 launcher: scale inside the GEMM, no materialized copies."""
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = (triton.cdiv(M, BLOCK_M) * triton.cdiv(N, BLOCK_N),)
    block_scaled_matmul_kernel[grid](
        a,
        b,
        c,
        M,
        N,
        K,
        float(scale_a),
        float(scale_b),
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_K=BLOCK_K,
    )
    return c


### Pitfalls

- Scalar teaching scales ≠ per-block FP8 microscales.
- Materializing full dequant tensors wastes HBM vs in-loop scaling.
- Accumulating in FP8 instead of FP32: the error explodes.
- Clipping: if $|x|>x_{\max}$, the error bound (0.7) fails.

### Self-check

1. Prove $C=(s_AA)(s_BB)=(s_As_B)AB$ for scalars.
2. Derive the per-block formula (0.4) and identify where the scale multiplies.
3. Where would per-block scales be read in the $K$-loop?
4. Why does FP8 need scales at all? Derive (0.7).
5. Compute the scale overhead for block size $32$ with FP32 scales.


In [ ]:
def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    a = torch.randn(32, 16)
    b = torch.randn(16, 24)
    sa, sb = 0.5, 3.0          # product 1.5, not 1: a kernel that ignores scales fails
    ref = torch.einsum("ik,kj->ij", a * sa, b * sb)
    assert torch.allclose(
        naive_block_scaled_matmul_pseudo(a, b, sa, sb), ref, atol=1e-5
    )


def test_correctness() -> None:
    require_gpu()
    m, k, n = 256, 128, 512
    a = torch.randn(m, k, device=DEVICE, dtype=torch.float16)
    b = torch.randn(k, n, device=DEVICE, dtype=torch.float16)
    sa = torch.tensor(0.5, device=DEVICE, dtype=torch.float16)
    sb = torch.tensor(3.0, device=DEVICE, dtype=torch.float16)
    ref = torch.matmul(a.float() * sa, b.float() * sb)
    c_ref = block_scaled_matmul_ref(a, b, sa, sb)
    c_tl = block_scaled_matmul_triton(a, b, float(sa), float(sb))
    assert torch.allclose(c_ref.float(), ref.float(), atol=1e-2, rtol=1e-2)
    assert torch.allclose(c_tl.float(), ref.float(), atol=1e-2, rtol=1e-2)
    print("block-scaled OK")


In [ ]:
def test_fused_block_scaled() -> None:
    """Method 03 (scale inside GEMM) vs Method 01 (materialized)."""
    require_gpu()
    torch.manual_seed(0)
    m, k, n = 256, 128, 512
    a = torch.randn(m, k, device=DEVICE, dtype=torch.float16)
    b = torch.randn(k, n, device=DEVICE, dtype=torch.float16)
    sa, sb = 0.5, 3.0
    ref = torch.matmul(a.float() * sa, b.float() * sb)
    out = block_scaled_matmul_fused(a, b, sa, sb)
    assert torch.allclose(out.float(), ref.float(), atol=1e-2, rtol=1e-2)
    print("fused block-scaled matmul OK")
